# Pulizia dei dati e pipeline con scikit-learn 

*Docenti*: Luca Calatroni, Lorenzo Rosasco

Obiettivi del notebook sono:
- pulizia dei dati di un dataset: variabili numeriche e categoriche, valori mancanti;
- preprocessing;
- `Pipeline` e `ColumnTransformer`;
- capire la distinzione tra **preprocessing** e **modello**

## 1. Caricamento del dataset e test naive

Useremo il dataset Titanic tramite `fetch_openml` (richiamo da OpenML)


In [1]:
from sklearn.datasets import fetch_openml

# Utilizziamo il dataset 'Titanic' che contiene in formato tabulare numerose informazioni sui passeggeri della famosa nave naufragata

titanic = fetch_openml(name="titanic", version=1, as_frame=True)
df = titanic.frame

df.head()


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


- Quante colonne vedete?
- Quale colonna pensate sia il target?
- Come classificheste le colonne in numeriche/categoriche?
- Notate valori mancanti?


In [2]:
print("Shape del dataset:", df.shape)
df.info()

Shape del dataset: (1309, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   object  
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   object  
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    object  
 10  embarked   1307 non-null   category
 11  boat       486 non-null    object  
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    object  
dtypes: category(3), float64(3), int64(3), object(5)
memory usage: 116.8+ KB


In [3]:
# Contiamo i valori mancanti
df.isnull().sum().sort_values(ascending=False)


body         1188
cabin        1014
boat          823
home.dest     564
age           263
embarked        2
fare            1
pclass          0
survived        0
name            0
sex             0
sibsp           0
parch           0
ticket          0
dtype: int64

- Quali colonne hanno molti valori mancanti?
- Se provassimo a fare `fit()` subito, cosa vi aspettate? Perché?


Estraiamo la colonna `survived` dal dataset e utilizziamola come target (vettore di etichette binarie). La colonna indica se il passeggero è sopravvissuto (1) oppure no (0).


In [4]:
print(df["survived"].value_counts(dropna=False))

survived
0    809
1    500
Name: count, dtype: int64


In [5]:
# Creazione del dataset

# X: tutte le colonne tranne survived
# y: la colonna survived

# TO DO

X = df.drop(columns=["survived"])
y = df["survived"]

print("Shape di X:", X.shape)
print("Shape di y:", y.shape)


Shape di X: (1309, 13)
Shape di y: (1309,)


Proviamo ad allenare direttamente un modello di nostra scelta (regressione logistica).


In [6]:
from sklearn.linear_model import LogisticRegression # TO DO: choose logistic regression

try:
    model = LogisticRegression(max_iter=5000)
    model.fit(X, y)  # use the entire X and y for simplicity
except Exception as e:
    print("Errore durante il fit:")
    print(e)


Errore durante il fit:
could not convert string to float: 'Allen, Miss. Elisabeth Walton'


- Perché il modello fallisce? 

## 2. Pulizia dei dati e preprocessing

In [7]:
# Identifichiamo le colonne numeriche (intere e non) e quelle categoriche

numeric_features = X.select_dtypes(include=['int64','float64']).columns  # TO DO: scrivere il formato
categorical_features = X.select_dtypes(include=['object','category','bool']).columns

print("Feature numeriche:")
print(list(numeric_features))

print("\nFeature categoriche:")
print(list(categorical_features))


Feature numeriche:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'body']

Feature categoriche:
['name', 'sex', 'ticket', 'cabin', 'embarked', 'boat', 'home.dest']


- Ha senso fare una media di una colonna categorica?
- Ha senso passare una stringa direttamente ad una regressione logistica?

## Strategie di preprocessing

**Feature numeriche**
1. sostituzione dei valori mancanti tramite regola (imputazione). Per semplicità utilizziamo la media di ogni feature $x_j$
$
\mu_j = \frac{1}{n} \sum_{i=1}^n x_{ij}
$

Sostituiamo facendo:
$$
x_{ij}^{(\text{imputed})} =
\begin{cases}
x_{ij} & \text{se osservato} \\
\mu_j & \text{se mancante}
\end{cases}
$$


2. conversione delle feature numeriche su scale confrontabili (scaling) tramite standardizzazione. Per ogni feature $x_j$:

$$
x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

dove:
- $ \mu_j $ è la media della feature  
- $ \sigma_j $ è la deviazione standard  

Dopo la trasformazione:
- media = 0  
- varianza = 1  

**Nota**: $ \( \mu_j \) $ e  $ \( \sigma_j \) $ vengono stimati **sul training set**.

**Feature categoriche**
1. imputazione dei valori mancanti con il valore più frequente
$$
x^{(\text{imputed})} = \arg\max_{c} \; \text{freq}(c)
$$
dove $ c $ varia tra le categorie osservate.
2. codifica in colonne binarie (codifica One-Hot). Data una variabile categorica con \( K \) categorie:
$
c \in \{c_1, c_2, \dots, c_K\}
$
viene trasformata in un vettore binario:
$
c_k \rightarrow e_k \in \{0,1\}^K
$
dove:

$$
(e_k)_i =
\begin{cases}
1 & \text{se } i = k \\
0 & \text{altrimenti}
\end{cases}
$$

Esempio:
$$\
\text{red} \rightarrow [1,0,0], \quad
\text{blue} \rightarrow [0,1,0], \quad
\text{green} \rightarrow [0,0,1]
$$

Questa trasformazione evita di introdurre un ordine artificiale tra le categorie che se verificherebbe se associassimo dei valori scalari.


In scikit-learn è molto comodo costruire pipeline separate per gruppi di colonne diversi.

In [8]:
# Carichiamo le librerie necessarie

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [9]:
#Trattiamo diversamente le colonne numeriche e quelle categoriche

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))  # evita errori se nel test set compare una categoria non vista durante il training
])


- Quando ha senso usare `mean` per le variabili numeriche? É rischioso usare `most_frequent` per le categoriche? Cosa proponete?
- Perché facciamo scaling solo sulle numeriche?
- Perché `OneHotEncoder` crea più colonne invece di assegnare numeri alle categorie?

Utilizzo di `ColumnTransformer`: permette di dire a scikit-learn

- su queste colonne fai un certo preprocessing;
- su queste altre colonne fai un preprocessing diverso.

In [14]:
# Preprocessing dei dati in base alla loro categoria

# TO DO: associare il preprocessing relativo

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


## 3. Costruzione della pipeline completa

Ora combiniamo:
1. preprocessing
2. modello

In [15]:
# TO DO: compilare con il nome del metodo di preprocessing + modello opportuno con iperparametri

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=5000))
])

pipeline


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Procediamo come visto la volta scorsa per fissare il fit del modello sui dati in questione. La procedure di preprocessing sono automaticamente codificate.

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # TO DO: costruire train/test sets

print("Shape X_train:", X_train.shape)
print("Shape X_test :", X_test.shape)


Shape X_train: (1047, 13)
Shape X_test : (262, 13)


- Perché non dobbiamo imputare e scalare usando tutto il dataset prima dello split, ma solo il training set?
- Che tipo di errore concettuale si commetterebbe?


Il comando `fit()` lavora sull'intera pipeline:
- calcola i parametri del preprocessing ed effettua imputazione/scaling;
- addestra il modello finale sui dati pre-processati

In [17]:
pipeline.fit(X_train,y_train)  # TO DO : trainare il modello

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


- Che cosa fa `fit()` in questa pipeline?
- Quali oggetti stanno imparando qualcosa dai dati?
- Solo il modello finale impara, oppure anche il preprocessing?


In [18]:
# Controlliamo le features dopo il pre-processing

X_train_prepared = preprocessor.fit_transform(X_train)
print(X_train_prepared.shape)

(1047, 2333)


In [19]:
# --- COLONNA CATEGORICA: sex ---

# estraiamo la colonna sex come array 2D
sex = df["sex"].to_numpy().reshape(-1, 1)

print("Colonna originale (sex):")
print(sex[:5])

encoder = OneHotEncoder(sparse_output=False)
sex_encoded = encoder.fit_transform(sex)

print("\nCategorie trovate:")
print(encoder.categories_)

print("\nColonna dopo One-Hot Encoding:")
print(sex_encoded[:5])

Colonna originale (sex):
[['female']
 ['male']
 ['female']
 ['male']
 ['female']]

Categorie trovate:
[array(['female', 'male'], dtype=object)]

Colonna dopo One-Hot Encoding:
[[1. 0.]
 [0. 1.]
 [1. 0.]
 [0. 1.]
 [1. 0.]]


In [20]:
# --- COLONNA NUMERICA: age ---

age = df["age"].to_numpy().reshape(-1, 1)

print("\nColonna originale (age):")
print(age[10:20])

# imputazione (media)
imputer = SimpleImputer(strategy="mean")
age_imputed = imputer.fit_transform(age)

# scaling
scaler = StandardScaler()
age_scaled = scaler.fit_transform(age_imputed)

print("\nColonna dopo imputazione e scaling:")
print(age_scaled[10:20])


Colonna originale (age):
[[47.]
 [18.]
 [24.]
 [26.]
 [80.]
 [nan]
 [24.]
 [50.]
 [32.]
 [36.]]

Colonna dopo imputazione e scaling:
[[ 1.32928228e+00]
 [-9.22571740e-01]
 [-4.56670909e-01]
 [-3.01370632e-01]
 [ 3.89173685e+00]
 [ 2.75868709e-16]
 [-4.56670909e-01]
 [ 1.56223269e+00]
 [ 1.64530199e-01]
 [ 4.75130753e-01]]


Ora possiamo usare la pipeline per fare predizioni sul test set.

In [ ]:
# TO DO: effettuare predizioni sul test set

y_pred = 

print("Prime 10 predizioni:", y_pred[:10])
print("Prime 10 etichette:", y_test.iloc[:10].values)


## 4. Cambio modello con stesso preprocessing

Una volta fissato il preprocessing, possiamo cambiare facilmente il modello finale.

In [ ]:
# Applichiamo kNN

from sklearn.neighbors import ...

pipeline_knn = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ...)
])

pipeline_knn.fit(....,....)
y_pred_knn = pipeline_knn.predict(...)

print("Prime 10 predizioni del Decision Tree:", y_pred_knn[:10])


In [ ]:
print("Prime 10 predizioni Logistic Regression:", y_pred[:10])
print("Prime 10 predizioni kNN      :", y_pred_knn[:10])
